In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    max,
    when,
    col,
    round
)

# Read Silver
silver_df = spark.table("workspace.default.silver_transactions")


# ============================================================
# 1. Gold: Transaction Fact Table
# ============================================================

fact_transactions = silver_df.select(
    "Time",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
    "V7",
    "V8",
    "V9",
    "V10",
    "V11",
    "V12",
    "V13",
    "V14",
    "V15",
    "V16",
    "V17",
    "V18",
    "V19",
    "V20",
    "V21",
    "V22",
    "V23",
    "V24",
    "V25",
    "V26",
    "V27",
    "V28",
    "Amount",
    "Class",
    "_ingestion_time"
)

(
    fact_transactions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.fact_transactions")
)


# ============================================================
# 2. Gold: Overall Fraud Summary
# ============================================================

fraud_summary = silver_df.agg(
    count("*").alias("total_transactions"),
    sum(
        when(col("Class") == 1, 1).otherwise(0)
    ).alias("fraud_transactions"),
    sum(
        when(col("Class") == 0, 1).otherwise(0)
    ).alias("normal_transactions"),
    round(
        avg(when(col("Class") == 1, 1).otherwise(0)) * 100,
        4
    ).alias("fraud_rate_percent"),
    sum(
        when(col("Class") == 1, col("Amount")).otherwise(0)
    ).alias("fraud_amount"),
    sum("Amount").alias("total_transaction_amount")
)

(
    fraud_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.fraud_summary")
)


# ============================================================
# 3. Gold: Amount-based Fraud Summary
# ============================================================

amount_summary = (
    silver_df
    .groupBy("Class")
    .agg(
        count("*").alias("transaction_count"),
        round(sum("Amount"), 2).alias("total_amount"),
        round(avg("Amount"), 2).alias("average_amount"),
        round(max("Amount"), 2).alias("maximum_amount")
    )
)

(
    amount_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.amount_summary")
)


print("Gold tables created successfully!")

Gold tables created successfully!


In [0]:
print("Fact transactions:")
spark.table("workspace.default.fact_transactions").count()

Fact transactions:


283726

In [0]:
print("Fraud summary:")
spark.table("workspace.default.fraud_summary").display()

Fraud summary:


total_transactions,fraud_transactions,normal_transactions,fraud_rate_percent,fraud_amount,total_transaction_amount
283726,473,283253,0.1667,58591.38999999996,2.5102001679986306E7


In [0]:
print("Amount summary:")
spark.table("workspace.default.amount_summary").display()

Amount summary:


Class,transaction_count,total_amount,average_amount,maximum_amount
0,283253,2.504341029E7,88.41,25691.16
1,473,58591.39,123.87,2125.87
